# NYC Taxi Pipeline: Passo a Passo

Execute as células em ordem. Cada seção corresponde a uma etapa do pipeline.

```
[Raw] → [Bronze] → [Silver Yellow + Green] → [Gold] → [Queries]
```

## 0. Setup: sys.path

Resolve o caminho do projeto dinamicamente a partir da localização do notebook; funciona independente do usuário ou diretório.

In [0]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = "/Workspace" + "/".join(notebook_path.split("/")[:-2])

sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

Project root: /Workspace/Users/ewelimdsb1@gmail.com/NYC_taxi_pipeline


## 1. Inicializar Unity Catalog

Cria os schemas `nyc_taxi.raw`, `nyc_taxi.bronze`, `nyc_taxi.silver`, `nyc_taxi.gold` e os Volumes `nyc_taxi.raw.files` e `nyc_taxi.bronze.files`.  
**Executar apenas uma vez** - idempotente, não sobrescreve dados existentes.

In [0]:
from config.setup_catalog import setup_unity_catalog
setup_unity_catalog(spark)

[Setup] nyc_taxi.raw.files → /Volumes/nyc_taxi/raw/files/
[Setup] nyc_taxi.bronze.files → /Volumes/nyc_taxi/bronze/files/
[Setup] Schema nyc_taxi.silver created
[Setup] Schema nyc_taxi.gold created


## 2. Download dos dados brutos

Baixa 20 parquets do NYC TLC (5 meses × 4 tipos: yellow, green, fhv, fhvhv).  
Aguarde ~3-4 minutos.

In [0]:
from src.ingestion.download_raw import download_all
download_all()

  OK: 47.7 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-01.parquet
  OK: 47.7 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-02.parquet
  OK: 56.1 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-03.parquet
  OK: 54.2 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-04.parquet
  OK: 58.7 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-05.parquet
  OK: 1.4 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-01.parquet
  OK: 1.5 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-02.parquet
  OK: 1.7 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-03.parquet
  OK: 1.6 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-04.parquet
  OK: 1.7 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-05.parquet
  OK: 11.3 MB → /Volumes/nyc_taxi/raw/files/fhv/fhv_tripdata_2023-01.parquet
  OK: 12.8 MB → /Volumes/nyc_taxi/raw/files/fhv/fhv_tripdata_2023-02.parquet
  OK: 15.0 MB → /Volumes/nyc_ta

## 3. Ingestão Bronze

Lê os parquets brutos, normaliza tipos e grava em Delta Lake particionado por `year/month`.  
Adiciona colunas de lineage: `_ingested_at`, `_source_file`, `_source_system`.

In [0]:
from src.ingestion.bronze_ingest import ingest_all_bronze
ingest_all_bronze(spark)

[Bronze] yellow 2023-01: 3,066,766 records
[Bronze] yellow 2023-02: 2,913,955 records
[Bronze] yellow 2023-03: 3,403,766 records
[Bronze] yellow 2023-04: 3,288,250 records
[Bronze] yellow 2023-05: 3,513,649 records
[Bronze] green 2023-01: 68,211 records
[Bronze] green 2023-02: 64,809 records
[Bronze] green 2023-03: 72,044 records
[Bronze] green 2023-04: 65,392 records
[Bronze] green 2023-05: 69,174 records
[Bronze] fhv 2023-01: 1,114,320 records
[Bronze] fhv 2023-02: 1,110,797 records
[Bronze] fhv 2023-03: 1,328,242 records
[Bronze] fhv 2023-04: 1,246,479 records
[Bronze] fhv 2023-05: 1,385,826 records
[Bronze] fhvhv 2023-01: 18,479,031 records
[Bronze] fhvhv 2023-02: 17,960,971 records
[Bronze] fhvhv 2023-03: 20,413,539 records
[Bronze] fhvhv 2023-04: 19,144,903 records
[Bronze] fhvhv 2023-05: 19,847,676 records


## 4. Transformação Silver

Aplica schema canônico, filtra nulos e dados duplicados.  
Cria `nyc_taxi.silver.yellow_trips` e `nyc_taxi.silver.green_trips` com liquid clustering por `pickup_datetime`.

In [0]:
from src.transformation.silver_yellow import build_silver_yellow
from src.transformation.silver_green import build_silver_green

build_silver_yellow(spark)
build_silver_green(spark)

[Silver Yellow] 16,039,728 records written → nyc_taxi.silver.yellow_trips
[Silver Yellow] Rodando OPTIMIZE (liquid clustering) ...
[Silver Yellow] OPTIMIZE concluído.
[Silver Green] 338,681 records written → nyc_taxi.silver.green_trips
[Silver Green] Rodando OPTIMIZE (liquid clustering) ...
[Silver Green] OPTIMIZE concluído.


## 5. Build da Gold

Une Silver Yellow + Green via `unionByName`, filtra datas corrompidas e grava `nyc_taxi.gold.trips` com liquid clustering por `pickup_datetime`.

In [0]:
from src.transformation.gold_trips import build_gold_trips
build_gold_trips(spark)

[Gold] 16,378,337 records written → nyc_taxi.gold.trips
  Yellow: 16,039,661
  Green:  338,676
[Gold] Rodando OPTIMIZE (liquid clustering) ...
[Gold] OPTIMIZE concluído.


## 6. Validação de qualidade

In [ ]:
from src.quality.checks import validate_layer

gold_df = spark.table("nyc_taxi.gold.trips")
validate_layer(gold_df, "Gold")

## 7. Verificação de idempotência

Re-execute os passos 3-5 e rode a célula abaixo. O count deve ser idêntico.

In [ ]:
print(spark.sql("SELECT COUNT(*) FROM nyc_taxi.gold.trips").collect()[0][0])

## 8. Análise dos resultados

Pipeline concluído. Para explorar os resultados, abra:

- **`analysis/results.ipynb`**: queries obrigatórias com visualizações e insights (Yellow vs Green, demanda por hora, audit trail)
- **`analysis/query1_avg_total_amount.sql`** e **`query2_avg_passengers_by_hour.sql`**: queries brutas para rodar direto no editor SQL do Databricks